In [1]:
import pandas as pd
import json

def add_coordinates(data, bases_json_path, location_column):
    """
    Adds latitude and longitude columns to any dataset using locations from bases.json.

    Parameters:
        data (str or pd.DataFrame): Path to CSV/JSON/Excel file OR a DataFrame.
        bases_json_path (str): Path to bases.json file.
        location_column (str): Name of the column containing location names.

    Returns:
        pd.DataFrame: Updated DataFrame with latitude and longitude columns.
    """

    # Load bases.json (location → coordinates)
    with open(bases_json_path, "r") as f:
        bases = json.load(f)

    # Build mapping dictionary
    coord_map = {
        item["name"].strip().lower(): (item["lat"], item["lon"])
        for item in bases
    }

    # Load data if file path is provided
    if isinstance(data, str):
        if data.endswith(".csv"):
            df = pd.read_csv(data)
        elif data.endswith(".json"):
            df = pd.read_json(data)
        elif data.endswith(".xlsx"):
            df = pd.read_excel(data)
        else:
            raise ValueError("Unsupported file format")
    else:
        df = data.copy()

    # Normalize column for matching
    df["__key"] = df[location_column].astype(str).str.strip().str.lower()

    # Map coordinates
    df["latitude"] = df["__key"].map(lambda x: coord_map.get(x, (None, None))[0])
    df["longitude"] = df["__key"].map(lambda x: coord_map.get(x, (None, None))[1])

    # Drop helper column
    df.drop(columns=["__key"], inplace=True)

    return df


In [2]:
# State file paths
# Financial Statements
financial_statements_1_path = 'data/Financial Statements/Financial Statements Format-1.csv'
financial_statements_2_europe_path = 'data/Financial Statements/Financial Statements Format-2 Europe.csv'
financial_statements_2_korea_path = 'data/Financial Statements/Financial Statements Format-2 Korea.csv'
financial_statements_2_japan_path = 'data/Financial Statements/Financial Statements Format-2 Japan.csv'
financial_statements_3_europe_path = 'data/Financial Statements/Financial Statements Format-3 Europe.csv'
financial_statements_3_korea_path = 'data/Financial Statements/Financial Statements Format-3 Korea.csv'
financial_statements_3_japan_path = 'data/Financial Statements/Financial Statements Format-3 Japan.csv'
financial_statements_4_path = 'data/Financial Statements/Financial Statements Format-4.csv'

# Navy Revenue
navy_revenue_1_path = 'data/Navy Revenue/Navy Revenue Report FY16-FY24-1 Format 1.csv'
navy_revenue_2_path = 'data/Navy Revenue/Navy Revenue Report FY20-FY24-1 Format 2.csv'

# Marine Revenue
marine_revenue_1_path = 'data/Marine Revenue/Marine Revenue FY20-FY24 Format 1.csv'
marine_revenue_2_path = 'data/Marine Revenue/Marine Revenue FY20-FY24 Format 2.csv'

# Base Location
bases_file_path = 'data/bases.json'

In [3]:
# Import and read csv files
# Financial Statements
fs1 = pd.read_csv(financial_statements_1_path)
fs2e = pd.read_csv(financial_statements_2_europe_path)
fs2k = pd.read_csv(financial_statements_2_korea_path)
fs2j = pd.read_csv(financial_statements_2_japan_path)
fs3e = pd.read_csv(financial_statements_3_europe_path)
fs3k = pd.read_csv(financial_statements_3_korea_path)
fs3j = pd.read_csv(financial_statements_3_japan_path)
fs4 = pd.read_csv(financial_statements_4_path)

# Navy Revenue
nr1 = pd.read_csv(navy_revenue_1_path)
nr2 = pd.read_csv(navy_revenue_2_path)

# Marine Revenue
mr1 = pd.read_csv(marine_revenue_1_path)
mr2 = pd.read_csv(marine_revenue_2_path)

# Bases Locations
bases = pd.read_json(bases_file_path)

In [12]:
fs4 = add_coordinates(fs4, bases_file_path, "Locations")
fs4.to_csv(financial_statements_4_path)